## Setting up the data and tools

In [1]:
# Setting up the python/jupyter environment
import pandas as pd
import ROOT
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
import tabulate

%matplotlib inline
plt.rcParams["figure.figsize"] = (20,10)
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

Welcome to JupyROOT 6.28/00


In [2]:
import json
import os
from pathlib import Path
with open(Path(os.getcwd()) / ".." / "config" / "config.json") as f:
    cfg = json.load(f)
data_prefix = cfg["DATA_DIR"]
print(f"Loading data from {data_prefix}")

Loading data from root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/ap_post_process/output


In [3]:
def get_filenames(datatype, eventtype, sign):
    polarities = [ "magdown", "magup" ]
    filenames = [ f"{data_prefix}/rds_finalnoBYsep_{datatype}_{eventtype}_{polarity}_{sign}.root" 
                      for polarity in polarities ]
    return filenames
filenames = get_filenames("2012", "23903000", "rs")
print(filenames)
rdf = ROOT.RDataFrame("DecayTree", filenames)

['root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/ap_post_process/output/rds_finalnoBYsep_2012_23903000_magdown_rs.root', 'root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/ap_post_process/output/rds_finalnoBYsep_2012_23903000_magup_rs.root']


In [4]:
print(f"Loaded rdf with {rdf.Count().GetValue()} entries")

Loaded rdf with 1440058 entries


In [5]:
import categories4 as f
rdf2 = f.add_categories_and_filter(rdf, apply_BDT_Iso_cut=True, apply_PIDK_cut=True)
print(f"After BDT_Iso and PIDK_cuts: {rdf2.Count().GetValue()} entries")

After BDT_Iso and PIDK_cuts: 1207697 entries


In [6]:
rdf3 = rdf2.Filter("B_Y_SEP < -4.5")

In [7]:
print(f"After B_Y_SEP cut: {rdf3.Count().GetValue()} entries")

After B_Y_SEP cut: 250022 entries


In [10]:
rdf4 = rdf3.Filter("q2_2 > 0 && B_M < 5000")
print(f"q2_2 neg: {rdf4.Count().GetValue()} entries")

q2_2 neg: 233110 entries


In [9]:
250022 - 7282

242740

In [ ]:
#!rm temp_withbdtallcols.root

In [ ]:
%%time
to_keep =  [ "category", "simplified", "B_M", "B_Y_SEP", "Xc_signal_Ypis_displaced_fromBs_fromTau", "fromY_from_B_vertex"]
to_keep += [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_M",
    # log(abs(PBsn))
    # log(abs(PBv/B_P))
    # log(abs(PBvn/B_P))
    # log(abs((PBsn-PBvn)/PBvn))
    # log(sqrt(abs(mDs2vn)))
    # log(Y_PE)
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass", 
    "q2_2",
    "tauY_2",
    "B_BPVVDR", 
    "mN2v",
    "B_Y_SEP",
    "B_correctedMass",
    "eventIndex",
    "PBsn",
    "PBv",
    "PBvn",
    "B_P",
    "mDs2vn",
    "Y_PE",
    "max_m2pi",
    "min_m2pi"
]
all_cols = list(rdf2.GetColumnNames())
print(f"Missing columns: {[c for c in to_keep if c not in all_cols]}")
if not os.path.exists("temp_withbdtallcols.root"):
    rdf2 = f.add_categories_and_filter(rdf) # Adds 
    print(f"Loaded rdf and filtered: {rdf2.Count().GetValue()} entries")
    rdf2.Snapshot("DecayTree", "temp_withbdtallcols.root", to_keep)

In [ ]:
df2 = pd.DataFrame(rdf2.Cache( [ "category", "simplified", "B_M", "B_Y_SEP", "Xc_signal_Ypis_displaced_fromBs_fromTau", "fromY_from_B_vertex"]).AsNumpy())

In [ ]:
df2.hist("B_M", bins=300, histtype='step')

In [ ]:
df2.hist("B_Y_SEP", bins=300, histtype='step', range=[-15, 5])

# Double checking old file

In [ ]:
rdfsep = ROOT.RDataFrame("DecayTree", "temp_v1_Ferara.root")

In [ ]:
print(f"{rdfsep.Count().GetValue()} entries")

In [ ]:
dfsep = pd.DataFrame(rdfsep.Cache().AsNumpy())

In [ ]:
dfsep.hist("B_M", bins=300)

In [ ]:
dfsep.hist("B_Y_SEP", bins=300, range=[-15,5])